In [4]:
import pandas as pd
from pathlib import Path

ROOT = Path("..")
sii = pd.read_excel(ROOT / "data/external/datos_sii.xlsx")

print(sii.shape)
print(sii.columns.tolist())
sii.head(3)

(81611, 19)
['Código SII', 'Año', 'Tipo', 'Marca', 'Modelo', 'Versión', 'Puertas', 'Cilindrada (CC)', 'Potencia (HP)', 'Combustible', 'Transmisión', 'Marchas', 'Tracción', 'País', 'Equipamiento Antiguo', 'Equipamiento', 'Tasación 2026', 'Permiso 2026', 'Beneficio Ley 21.505']


,Código SII,Año,Tipo,Marca,Modelo,Versión,Puertas,Cilindrada (CC),Potencia (HP),Combustible,Transmisión,Marchas,Tracción,País,Equipamiento Antiguo,Equipamiento,Tasación 2026,Permiso 2026,Beneficio Ley 21.505
0,CB0110001,2018,Cabriolet,ASTON MARTIN,DB11,4.0 LTS CONVERTIBLE 2P T/A,2.0,4000,510.0,Bencina,Automática,8.0,4x2 (2WD),INGLATERRA,NaN,"AAF, ABP, ABS, ACA, AED, AEL, AIN, ALC, ASR, B...",102510925,4173560,NaN
1,CB0110001,2019,Cabriolet,ASTON MARTIN,DB11,4.0 LTS CONVERTIBLE 2P T/A,2.0,4000,510.0,Bencina,Automática,8.0,4x2 (2WD),INGLATERRA,NaN,"AAF, ABP, ABS, ACA, AED, AEL, AIN, ALC, ASR, B...",113963741,4688937,NaN
2,CB0110001,2020,Cabriolet,ASTON MARTIN,DB11,4.0 LTS CONVERTIBLE 2P T/A,2.0,4000,510.0,Bencina,Automática,8.0,4x2 (2WD),INGLATERRA,NaN,"AAF, ABP, ABS, ACA, AED, AEL, AIN, ALC, ASR, B...",120451818,4980901,NaN


In [5]:
# Tipos y nulos
info = pd.DataFrame({
    "dtype": sii.dtypes.astype(str),
    "nulos": sii.isna().sum(),
    "pct_nulos": (sii.isna().mean() * 100).round(2),
    "n_unicos": sii.nunique(dropna=True)
}).sort_values(["pct_nulos", "n_unicos"], ascending=[False, False])

info

,dtype,nulos,pct_nulos,n_unicos
Beneficio Ley 21.505,str,80950,99.19,1
Equipamiento,str,68228,83.60,3430
Potencia (HP),float64,67540,82.76,398
País,str,63860,78.25,27
Marchas,float64,63445,77.74,9
Tracción,str,61840,75.77,3
Equipamiento Antiguo,str,32717,40.09,3
Puertas,float64,15938,19.53,5
Versión,str,6,0.01,18425
Tasación 2026,int64,0,0.00,46839


In [6]:
# Duplicados exactos
dup_exactos = sii.duplicated().sum()
print("Duplicados exactos:", dup_exactos)

Duplicados exactos: 0


In [7]:
# Si existe código SII, revisar unicidad
col_codigo = [c for c in sii.columns if "codigo" in c.lower() and "sii" in c.lower()]
print(col_codigo)

[]


In [8]:
# Revisar cobertura de variables técnicas candidatas
candidatas = [
    c for c in sii.columns
    if any(k in c.lower() for k in [
        "marca", "modelo", "version", "ano", "año",
        "cilindrada", "potencia", "traccion", "puertas",
        "combustible", "transmision", "carroceria", "tipo"
    ])
]
sii[candidatas].head()

,Año,Tipo,Marca,Modelo,Puertas,Cilindrada (CC),Potencia (HP),Combustible
0,2018,Cabriolet,ASTON MARTIN,DB11,2.0,4000,510.0,Bencina
1,2019,Cabriolet,ASTON MARTIN,DB11,2.0,4000,510.0,Bencina
2,2020,Cabriolet,ASTON MARTIN,DB11,2.0,4000,510.0,Bencina
3,2021,Cabriolet,ASTON MARTIN,DB11,2.0,4000,510.0,Bencina
4,2022,Cabriolet,ASTON MARTIN,DBS SUPERLEGGERA,2.0,5200,715.0,Bencina


In [9]:
# Frecuencia de filas por marca-modelo-año
# Ajusta nombres a tus columnas reales
cols = ["Marca", "Modelo", "Año"]
tmp = (
    sii.groupby(cols)
       .size()
       .reset_index(name="n_versiones")
       .sort_values("n_versiones", ascending=False)
)

tmp.head(20)

,Marca,Modelo,Año,n_versiones
20985,SUBARU,LEGACY,1996,62
20986,SUBARU,LEGACY,1997,60
20984,SUBARU,LEGACY,1995,56
17825,NISSAN,TERRANO,2003,49
8392,FOTON,TERRACOTA,2015,47
18433,PEUGEOT,307,2006,45
2586,BMW,X5,2013,43
18426,PEUGEOT,306,2000,42
18435,PEUGEOT,307,2008,41
20983,SUBARU,LEGACY,1994,40


In [10]:
# Qué agrega SII respecto a Yapo
cols_yapo = {
    "marca", "modelo", "anio", "combustible", "transmision"
}
cols_sii = set(c.lower() for c in sii.columns)

print("Posibles columnas nuevas útiles:")
for c in sorted(cols_sii):
    if all(x not in c for x in ["precio", "tasacion", "permiso"]):
        print("-", c)

Posibles columnas nuevas útiles:
- año
- beneficio ley 21.505
- cilindrada (cc)
- combustible
- código sii
- equipamiento
- equipamiento antiguo
- marca
- marchas
- modelo
- país
- potencia (hp)
- puertas
- tasación 2026
- tipo
- tracción
- transmisión
- versión


In [11]:
# ── Diagnóstico Kia ──────────────────────────────────────────────
kia_sin = df[(df["marca"] == "Kia") & (df["tiene_match"] == 0)]
print("Modelos Kia sin match en Yapo:")
print(kia_sin["modelo"].value_counts().head(20))
print()

# Ver cómo los tiene el SII
kia_sii = sii[sii["Marca"].str.lower().str.contains("kia", na=False)]
print(f"Registros Kia en SII: {len(kia_sii)}")
print("Modelos Kia en SII:")
print(kia_sii["Modelo"].value_counts().head(20))

NameError: name 'df' is not defined